# Generate Embeddings and Build a ChromaDB Knowledge Base

This notebook turns the processed research-paper pages into overlapping text chunks, generates local semantic embeddings, and stores the embeddings, chunk text, and source metadata in a persistent ChromaDB collection. It finishes with sample semantic-search queries and citation-ready result previews.

The notebook uses the recommended 400-word chunks with 80 words of overlap from `03_text_chunking.ipynb`.

## Load Processed Pages

The source file contains one cleaned record per paper page. Page metadata is carried into every chunk so retrieved passages can be cited back to the original PDF.

In [2]:
import json
from pathlib import Path

import pandas as pd

data_path = Path("../data/processed/cleaned_papers.json")
with data_path.open("r", encoding="utf-8") as file:
    records = json.load(file)

required_fields = {
    "paper_id", "title", "category", "pdf_url",
    "page_number", "cleaned_text"
}
assert isinstance(records, list) and records, "The input dataset must be a non-empty list"
assert all(required_fields <= set(record) for record in records)
assert all(record["cleaned_text"].strip() for record in records)

print(f"Loaded {len(records):,} cleaned page records from {data_path}")
pd.DataFrame(records).head(2)

Loaded 218 cleaned page records from ..\data\processed\cleaned_papers.json


,paper_id,title,category,pdf_url,page_number,cleaned_text
0,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,1,Retrieval-Augmented Generation for Knowledge-I...
1,2005.11401,Retrieval-Augmented Generation for Knowledge-I...,RAG,https://arxiv.org/pdf/2005.11401.pdf,2,The Divine Comedy (x) q Query Encoder q(x) MIP...


## Recreate the Recommended Chunks

Chunks are split on word boundaries and overlap so important technical phrases near a boundary remain retrievable.

In [3]:
def chunk_text(text, chunk_size=400, overlap=80):
    if not isinstance(text, str) or not text.strip():
        return []
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]
        if not chunk_words:
            break
        chunks.append({
            "text": " ".join(chunk_words),
            "start_word": start,
            "end_word": start + len(chunk_words) - 1,
            "word_count": len(chunk_words)
        })
        if start + chunk_size >= len(words):
            break
    return chunks

def chunk_records(records, chunk_size=400, overlap=80):
    chunked_records = []
    metadata_fields = ["paper_id", "title", "category", "pdf_url", "page_number"]
    for record in records:
        for chunk_index, chunk in enumerate(chunk_text(record["cleaned_text"], chunk_size, overlap)):
            chunked_records.append({
                **{field: record[field] for field in metadata_fields},
                "chunk_id": f"{record['paper_id']}-page-{record['page_number']}-chunk-{chunk_index}",
                "chunk_index": chunk_index,
                "chunk_size": chunk_size,
                "overlap": overlap,
                **chunk
            })
    return chunked_records

chunked_records = chunk_records(records)
chunked_df = pd.DataFrame(chunked_records)
assert chunked_df["chunk_id"].is_unique
assert chunked_df["text"].str.strip().all()
print(f"Created {len(chunked_df):,} chunks from {len(records):,} pages")
chunked_df[["chunk_id", "paper_id", "page_number", "word_count", "text"]].head(3)

Created 426 chunks from 218 pages


,chunk_id,paper_id,page_number,word_count,text
0,2005.11401-page-1-chunk-0,2005.11401,1,393,Retrieval-Augmented Generation for Knowledge-I...
1,2005.11401-page-2-chunk-0,2005.11401,2,400,The Divine Comedy (x) q Query Encoder q(x) MIP...
2,2005.11401-page-2-chunk-1,2005.11401,2,321,whereby both the generator and retriever are j...


## Create Embeddings and Persist ChromaDB

`all-MiniLM-L6-v2` is a compact local sentence-transformer suitable for a first semantic-search baseline. ChromaDB calls the embedding function during `add` and persists the resulting vectors under `data/chroma_db`.

In [4]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedding_model_name = "all-MiniLM-L6-v2"
embedding_function = SentenceTransformerEmbeddingFunction(model_name=embedding_model_name)

chroma_path = Path("../data/chroma_db")
chroma_path.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(chroma_path))
collection_name = "zayi_research_chunks"

try:
    client.delete_collection(collection_name)
except Exception:
    pass

collection = client.create_collection(
    name=collection_name,
    embedding_function=embedding_function,
    metadata={"description": "Zayi R&D research-paper chunks"}
)

documents = chunked_df["text"].tolist()
ids = chunked_df["chunk_id"].tolist()
metadatas = chunked_df[[
    "paper_id", "title", "category", "pdf_url",
    "page_number", "chunk_index", "chunk_size", "overlap"
]].to_dict(orient="records")

collection.add(ids=ids, documents=documents, metadatas=metadatas)
print(f"Stored {collection.count():,} embedded chunks in {chroma_path}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Stored 426 embedded chunks in ..\data\chroma_db


## Verify Stored Vectors, Text, and Metadata

This check confirms that the collection contains one record for every chunk and that ChromaDB can return the stored embedding, document text, and source metadata.

In [5]:
stored = collection.get(limit=1, include=["embeddings", "documents", "metadatas"])
stored_embedding = stored["embeddings"][0]
stored_document = stored["documents"][0]
stored_metadata = stored["metadatas"][0]

assert collection.count() == len(chunked_records)
assert stored_document and stored_metadata["paper_id"]
assert len(stored_embedding) > 0
print("Collection count:", collection.count())
print("Embedding dimensions:", len(stored_embedding))
print("Stored source:", stored_metadata["paper_id"], "page", stored_metadata["page_number"])
print("Text preview:", stored_document[:300])

Collection count: 426
Embedding dimensions: 384
Stored source: 2005.11401 page 1
Text preview: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela† †Facebook AI Research;‡University C


## Run Semantic-Search Queries

The result display includes the paper ID, page number, title, distance, and a passage preview. These fields provide the evidence trail needed for later RAG answers and citations.

In [6]:
def semantic_search(query, n_results=3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    rows = []
    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        rows.append({
            "distance": round(distance, 4),
            "paper_id": metadata["paper_id"],
            "page_number": metadata["page_number"],
            "title": metadata["title"],
            "pdf_url": metadata["pdf_url"],
            "passage": document[:500]
        })
    return pd.DataFrame(rows)

queries = [
    "How does retrieval-augmented generation combine parametric and non-parametric memory?",
    "What is the purpose of self-reflection in a retrieval-augmented generation system?",
    "How does dense passage retrieval find passages for open-domain question answering?"
]

search_results = {}
for query in queries:
    print("=" * 100)
    print("QUERY:", query)
    result_df = semantic_search(query, n_results=3)
    search_results[query] = result_df
    display(result_df)

QUERY: How does retrieval-augmented generation combine parametric and non-parametric memory?


,distance,paper_id,page_number,title,pdf_url,passage
0,0.3595,2005.11401,2,Retrieval-Augmented Generation for Knowledge-I...,https://arxiv.org/pdf/2005.11401.pdf,whereby both the generator and retriever are j...
1,0.3908,2310.11511,12,"Self-RAG: Learning to Retrieve, Generate, and ...",https://arxiv.org/pdf/2310.11511.pdf,"Woosuk Kwon, Zhuohan Li, Siyuan Zhuang, Ying S..."
2,0.3914,2005.11401,1,Retrieval-Augmented Generation for Knowledge-I...,https://arxiv.org/pdf/2005.11401.pdf,Retrieval-Augmented Generation for Knowledge-I...


QUERY: What is the purpose of self-reflection in a retrieval-augmented generation system?


,distance,paper_id,page_number,title,pdf_url,passage
0,0.3751,2310.11511,1,"Self-RAG: Learning to Retrieve, Generate, and ...",https://arxiv.org/pdf/2310.11511.pdf,LLMs or introduce unnecessary or off-topic pas...
1,0.4455,2310.11511,1,"Self-RAG: Learning to Retrieve, Generate, and ...",https://arxiv.org/pdf/2310.11511.pdf,"Preprint. SELF -RAG: L EARNING TO RETRIEVE , G..."
2,0.4886,2401.15884,10,Corrective Retrieval Augmented Generation (CRAG),https://arxiv.org/pdf/2401.15884.pdf,"while significantly enhancing performance, the..."


QUERY: How does dense passage retrieval find passages for open-domain question answering?


,distance,paper_id,page_number,title,pdf_url,passage
0,0.2663,2004.04906,1,Dense Passage Retrieval for Open-Domain Questi...,https://arxiv.org/pdf/2004.04906.pdf,Dense Passage Retrieval for Open-Domain Questi...
1,0.3225,2004.04906,2,Dense Passage Retrieval for Open-Domain Questi...,https://arxiv.org/pdf/2004.04906.pdf,"continuous space, such that it can retrieve ef..."
2,0.3372,2307.03172,3,Lost in the Middle: How Language Models Use Lo...,https://arxiv.org/pdf/2307.03172.pdf,models with longer input contexts is a trade-o...


## Retrieval Verification Summary

A successful run should show three results for every query, with non-empty text and citation metadata. Lower Chroma distance indicates a closer semantic match for the selected embedding model.

In [7]:
verification_rows = []
for query, result_df in search_results.items():
    assert len(result_df) == 3
    assert result_df["passage"].str.strip().all()
    assert result_df["paper_id"].str.strip().all()
    assert result_df["page_number"].notna().all()
    verification_rows.append({
        "query": query,
        "results_returned": len(result_df),
        "top_paper": result_df.iloc[0]["paper_id"],
        "top_page": int(result_df.iloc[0]["page_number"]),
        "top_distance": result_df.iloc[0]["distance"]
    })

verification_summary = pd.DataFrame(verification_rows)
print("Semantic retrieval verification passed.")
verification_summary

Semantic retrieval verification passed.


,query,results_returned,top_paper,top_page,top_distance
0,How does retrieval-augmented generation combin...,3,2005.11401,2,0.3595
1,What is the purpose of self-reflection in a re...,3,2310.11511,1,0.3751
2,How does dense passage retrieval find passages...,3,2004.04906,1,0.2663
